# Исходный учебный эксперимент
Исторические результаты сохранены. Ноутбук использует тестовую выборку как eval_set, не удаляет дубликаты; class_weights вычислены, но не переданы модели.
Для исправленного эксперимента с отдельной validation/test используйте ../train.py. Запускайте ноутбук из этой папки.

# Импортируем необходимые библиотеки

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_fscore_support, confusion_matrix, roc_curve
import seaborn as sns
import matplotlib.pyplot as plt
import shap
import pickle

# Загрузка набора данных

In [ ]:
# Load dataset
data = pd.read_csv('../data/diabetes_prediction_dataset.csv')

# Кодирование категориальных признаков

In [ ]:
# Encoding categorical features
data['gender'] = data['gender'].map({'Male': 1, 'Female': 0})
data['smoking_history'] = data['smoking_history'].astype('category').cat.codes


# Определение признаков (X) и целевой переменной (y)

In [ ]:
# Define features and target
X = data.drop(columns=['diabetes'])
y = data['diabetes']


# Вычисление весов классов для балансировки

In [ ]:
# Calculate class weights
class_weights = {0: len(y) / (2 * np.bincount(y)[0]), 
                 1: len(y) / (2 * np.bincount(y)[1])}

# Разделение данных на обучающую и тестовую выборки

In [ ]:

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# Обучение модели CatBoost

In [ ]:
# Train CatBoost model
original_model = CatBoostClassifier(iterations=100, learning_rate=0.1, depth=6, verbose=False, random_seed=42)
original_model.fit(X_train, y_train, eval_set=(X_test, y_test), verbose=False)

# Предсказания

In [ ]:
# Predictions
y_pred = original_model.predict(X_test)
y_pred_proba = original_model.predict_proba(X_test)[:, 1]

# Оценка ROC AUC

In [ ]:
# Metrics
roc_auc = roc_auc_score(y_test, y_pred_proba)
print("ROC AUC Score:", roc_auc)

In [ ]:
classification_report_str = classification_report(y_test, y_pred)
print("Отчет о классификации:\n", classification_report_str)

In [ ]:
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')
print(f"Точность: {precision:.2f}, Полнота: {recall:.2f}, F1-мера: {f1:.2f}")

# Матрица ошибок

In [ ]:
# Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Нет диабета', 'Диабет'], yticklabels=['Нет диабета', 'Диабет'])
plt.title('Матрица ошибок')
plt.xlabel('Предсказано')
plt.ylabel('Фактическое')
plt.show()

# Важность признаков

In [ ]:
# Feature Importance
feature_importances = original_model.get_feature_importance()
feature_names = X.columns

In [ ]:
sns.barplot(x=feature_importances, y=feature_names, palette='viridis')
plt.title('Важность признаков')
plt.xlabel('Важность')
plt.ylabel('Признаки')
plt.show()

# Корреляционная матрица

In [ ]:
# Correlation Matrix
correlation_matrix = data.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Корреляционная матрица признаков')
plt.show()

# ROC AUC кривая

In [ ]:
# Plot ROC AUC curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC кривая (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('Доля ложных срабатываний')
plt.ylabel('Доля истинных срабатываний')
plt.title('Кривая ROC AUC')
plt.legend(loc="lower right")
plt.show()

# Сохранение обученной модели в файл

In [ ]:
# Save the trained optimal model to a .cbm file
original_model.save_model("../models/original_model.cbm")
print("Model saved as 'diabetes_prediction_model.cbm'")